# HPI Prediction: ARIMA

This notebook forecasts House Price Index (HPI) for all 51 U.S. states using ARIMA, then evaluates performance on a 5-year holdout set (2020–2024).

## What is ARIMA?

**ARIMA** (AutoRegressive Integrated Moving Average) is a classical time series forecasting model defined by three parameters **(p, d, q)**:

- **AR (p)** — *AutoRegressive*: The model uses the previous *p* values of the series to predict the next value. This captures momentum — e.g., if HPI has been rising, it's likely to continue rising.
- **I (d)** — *Integrated*: The series is differenced *d* times to make it stationary (removing trends). For HPI, *d* = 1 means we model year-over-year changes rather than raw index values.
- **MA (q)** — *Moving Average*: The model uses the previous *q* forecast errors to correct its predictions. This helps the model adapt when recent predictions were off.

### Why ARIMA for HPI?

- HPI is a **univariate annual time series** with clear trends — exactly the type of data ARIMA is designed for.
- We use `pmdarima.auto_arima` to **automatically select** the best (p, d, q) order for each state via AIC minimization, so each state gets a tailored model.

**Metrics used**: MAE (Mean Absolute Error), RMSE (Root Mean Squared Error), and MAPE (Mean Absolute Percentage Error) — MAPE is the primary metric since it's scale-independent across states.

In [ ]:
import sys
import warnings
from pathlib import Path

# add src/ to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
from data_cleaning import prepare_state_hpi
from prediction import (
    split_train_test,
    run_arima_al
    l_states,
    compute_metrics,
)
from plot_utils import (
    plot_state_forecast, plot_forecast_grid,
    plot_forecast_choropleth, plot_forecast_choropleth_animated,
    plot_metrics_bar,
)

print("Imports loaded successfully.")

## 1. Load & Explore HPI Data

In [ ]:
raw_dir = project_root / "data" / "raw"
df = prepare_state_hpi(raw_dir)

print(f"Shape: {df.shape}")
print(f"States: {df['Abbreviation'].nunique()}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")
df.head(10)

In [ ]:
# quick look at HPI distribution across states
latest_year = df["Year"].max()
latest = df[df["Year"] == latest_year].sort_values("HPI", ascending=False)
print(f"\nHPI in {latest_year} — Top 5 and Bottom 5:")
print(latest[["Abbreviation", "State", "HPI"]].head())
print("...")
print(latest[["Abbreviation", "State", "HPI"]].tail())

## 2. ARIMA Forecasting (All States)

Uses `pmdarima.auto_arima` to automatically select (p,d,q) order per state. This takes ~1-3 minutes.

In [ ]:
%%time
arima_forecast, arima_eval, arima_models = run_arima_all_states(
    df, forecast_years=10, test_years=5
)
print(f"ARIMA forecasts: {len(arima_forecast)} rows")
print(f"ARIMA eval: {len(arima_eval)} rows")

## 3. Evaluate ARIMA on 2020–2024 Holdout

In [ ]:
metrics = compute_metrics(arima_eval)

print("=== ARIMA Metrics (Best 10 by MAPE) ===")
print(metrics.head(10).to_string(index=False))
print(f"\nMedian MAPE: {metrics['MAPE'].median():.2f}%")
print(f"Mean MAPE: {metrics['MAPE'].mean():.2f}%")
print(f"States with MAPE < 15%: {(metrics['MAPE'] < 15).sum()}/51")

print("\n=== Worst 10 by MAPE ===")
print(metrics.tail(10).to_string(index=False))

## 4. Visualizations

### 4a. Single State Forecast (California)

In [ ]:
fig = plot_state_forecast("CA", df, arima_forecast, eval_df=arima_eval)
fig.show()

### 4b. Multi-State Forecast Grid

In [ ]:
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=["CA", "TX", "NY", "FL", "IL", "WA"],
)
fig.show()

### 4c. Forecasts for All 51 States

In [ ]:
all_states = sorted(df["Abbreviation"].unique())
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=all_states,
)
fig.show()

### 4d. Choropleth Map (Predicted HPI in 2030)

In [ ]:
fig = plot_forecast_choropleth(arima_forecast, 2030)
fig.show()

### 4e. Animated Choropleth (History + Forecast)

In [ ]:
fig = plot_forecast_choropleth_animated(df, arima_forecast, history_years=5)
fig.show()

### 4f. MAPE by State Bar Chart

In [ ]:
fig = plot_metrics_bar(metrics, metric="MAPE", top_n=20)
fig.show()

## 5. Comprehensive Analysis

### 5a. ARIMA Model Orders Selected per State

`auto_arima` selects a different (p, d, q) order for each state. Let's examine what orders were chosen and what that tells us about each state's HPI dynamics.

In [ ]:
# extract ARIMA orders for each state
order_rows = []
for state, model in arima_models.items():
    p, d, q = model.order
    order_rows.append({"Abbreviation": state, "p": p, "d": d, "q": q,
                       "Order": f"({p},{d},{q})", "AIC": round(model.aic(), 2)})
orders_df = pd.DataFrame(order_rows).sort_values("Abbreviation")

print("ARIMA Orders Selected by auto_arima:\n")
print(orders_df.to_string(index=False))

print(f"\n--- Order Distribution ---")
print(orders_df["Order"].value_counts().to_string())
print(f"\nMost common d (differencing): {orders_df['d'].mode().iloc[0]} "
      f"({(orders_df['d'] == orders_df['d'].mode().iloc[0]).sum()}/51 states)")
print(f"Mean AIC: {orders_df['AIC'].mean():.1f}")

### 5b. Regional Performance Analysis

States can be grouped by U.S. Census region. Do certain regions have systematically higher or lower forecast errors?

In [ ]:
# define Census regions
regions = {
    "Northeast": ["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"],
    "Midwest": ["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"],
    "South": ["DE", "FL", "GA", "MD", "NC", "SC", "VA", "DC", "WV",
              "AL", "KY", "MS", "TN", "AR", "LA", "OK", "TX"],
    "West": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY",
             "AK", "CA", "HI", "OR", "WA"],
}
state_to_region = {s: r for r, states in regions.items() for s in states}
metrics_with_region = metrics.copy()
metrics_with_region["Region"] = metrics_with_region["Abbreviation"].map(state_to_region)

print("=== MAPE by Region ===\n")
region_stats = metrics_with_region.groupby("Region")["MAPE"].agg(["mean", "median", "min", "max", "count"])
region_stats.columns = ["Mean MAPE", "Median MAPE", "Best MAPE", "Worst MAPE", "N States"]
region_stats = region_stats.sort_values("Median MAPE")
print(region_stats.to_string())

print("\n\n=== Best & Worst State per Region ===\n")
for region in region_stats.index:
    r = metrics_with_region[metrics_with_region["Region"] == region].sort_values("MAPE")
    best = r.iloc[0]
    worst = r.iloc[-1]
    print(f"{region}:")
    print(f"  Best:  {best['Abbreviation']} ({best['State']}) — MAPE {best['MAPE']:.2f}%")
    print(f"  Worst: {worst['Abbreviation']} ({worst['State']}) — MAPE {worst['MAPE']:.2f}%")

### 5c. Error Pattern: Did the Model Consistently Under- or Over-Predict?

The 2020–2024 holdout period includes the COVID-era housing boom. Let's check whether ARIMA systematically underestimated the surge.

In [ ]:
import plotly.express as px
from scipy import stats

# compute signed error per state per year
eval_merged = arima_eval.merge(
    df[["Abbreviation", "Year", "HPI"]], on=["Abbreviation", "Year"], how="left"
)
eval_merged["Signed_Error"] = eval_merged["Predicted"] - eval_merged["HPI"]
eval_merged["Pct_Error"] = (eval_merged["Signed_Error"] / eval_merged["HPI"] * 100)

# year-by-year bias
print("=== Year-by-Year Forecast Bias (Predicted − Actual) ===\n")
yearly = eval_merged.groupby("Year").agg(
    Mean_Signed_Error=("Signed_Error", "mean"),
    Median_Signed_Error=("Signed_Error", "median"),
    Mean_Pct_Error=("Pct_Error", "mean"),
    MAPE=("Pct_Error", lambda x: x.abs().mean()),
    Underpredicted=("Signed_Error", lambda x: (x < 0).sum()),
).reset_index()
yearly["Year"] = yearly["Year"].astype(int)

for _, r in yearly.iterrows():
    direction = "UNDER" if r["Mean_Signed_Error"] < 0 else "OVER"
    print(f"  {r['Year']}:  Mean Error: {r['Mean_Signed_Error']:+7.1f}  "
          f"({r['Mean_Pct_Error']:+5.1f}%)  |  MAPE: {r['MAPE']:5.1f}%  |  "
          f"{direction}-predicted {int(r['Underpredicted'])}/51 states")

print(f"\nOverall: ARIMA underpredicted in {int(yearly['Underpredicted'].mean())}/51 states on average.")
print("The bias grows each year — the model missed the accelerating post-COVID surge.\n")

# --- Scatter plot: HPI growth (2019→2024) vs MAPE ---
growth_rates = []
for state in df["Abbreviation"].unique():
    state_df = df[df["Abbreviation"] == state].sort_values("Year")
    hpi_2019 = state_df[state_df["Year"] == 2019]["HPI"].values
    hpi_2024 = state_df[state_df["Year"] == 2024]["HPI"].values
    if len(hpi_2019) > 0 and len(hpi_2024) > 0:
        pct_growth = (hpi_2024[0] - hpi_2019[0]) / hpi_2019[0] * 100
        growth_rates.append({"Abbreviation": state, "Growth_Pct": round(pct_growth, 1)})
growth_df = pd.DataFrame(growth_rates)
scatter_df = metrics.merge(growth_df, on="Abbreviation")

# OLS trendline
slope, intercept, r_value, _, _ = stats.linregress(scatter_df["Growth_Pct"], scatter_df["MAPE"])

fig = px.scatter(
    scatter_df, x="Growth_Pct", y="MAPE", text="Abbreviation",
    labels={"Growth_Pct": "HPI Growth 2019→2024 (%)", "MAPE": "MAPE (%)"},
    title=f"Forecast Error vs Housing Boom Intensity (R² = {r_value**2:.2f})",
)
fig.update_traces(textposition="top center", marker=dict(size=8))
# add trendline
x_range = np.linspace(scatter_df["Growth_Pct"].min(), scatter_df["Growth_Pct"].max(), 100)
fig.add_scatter(x=x_range, y=intercept + slope * x_range, mode="lines",
                name=f"OLS (slope={slope:.2f})", line=dict(dash="dash", color="red"))
fig.update_layout(height=500, width=800)
fig.show()

print(f"Correlation: r = {r_value:.2f}, R² = {r_value**2:.2f}")
print(f"States with faster pandemic-era growth have proportionally higher MAPE.")

### 5d. Forecast Outlook: Predicted HPI Rankings in 2034

Using the fitted ARIMA models, we forecast HPI out to 2034 and rank states by predicted index value and growth rate.

In [ ]:
# --- 5d. Forecast outlook: predicted HPI rankings in 2034 ---
print("=" * 65)
print("FORECAST OUTLOOK: PREDICTED HPI RANKINGS IN 2034")
print("=" * 65)

forecast_2034 = arima_forecast[arima_forecast["Year"] == 2034].copy()
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
outlook = forecast_2034.merge(latest_actual, on="Abbreviation")
outlook["Growth_2024_2034_Pct"] = ((outlook["Predicted"] - outlook["HPI_2024"]) / outlook["HPI_2024"] * 100).round(1)

print("\nTop 10 states by predicted HPI in 2034:")
top = outlook.sort_values("Predicted", ascending=False).head(10)
for _, r in top.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print("\nBottom 10 states by predicted HPI in 2034:")
bot = outlook.sort_values("Predicted").head(10)
for _, r in bot.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print(f"\nHighest predicted growth:  {outlook.sort_values('Growth_2024_2034_Pct', ascending=False).iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].max():.1f}%)")
print(f"Lowest predicted growth:   {outlook.sort_values('Growth_2024_2034_Pct').iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].min():.1f}%)")
print(f"Median predicted growth:   +{outlook['Growth_2024_2034_Pct'].median():.1f}%")

### 5e. Confidence Interval Analysis

Wider confidence intervals indicate greater forecast uncertainty. Let's examine which states have the most and least certain forecasts.

In [ ]:
# --- 5e. Confidence interval width analysis ---
print("=" * 65)
print("CONFIDENCE INTERVAL ANALYSIS")
print("=" * 65)
print("\nWider 95% CI = more uncertainty in the forecast.\n")

ci_analysis = arima_forecast.copy()
ci_analysis["CI_Width"] = ci_analysis["CI_Upper"] - ci_analysis["CI_Lower"]

# CI width in 2034 (last forecast year)
ci_2034 = ci_analysis[ci_analysis["Year"] == 2034].sort_values("CI_Width", ascending=False)
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
ci_2034 = ci_2034.merge(latest_actual, on="Abbreviation")
ci_2034["CI_Width_Pct"] = (ci_2034["CI_Width"] / ci_2034["HPI_2024"] * 100).round(1)

print("States with widest 95% CI in 2034 (most uncertain):")
for _, r in ci_2034.head(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

print("\nStates with narrowest 95% CI in 2034 (most confident):")
for _, r in ci_2034.tail(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

# CI width growth over forecast horizon
print("\nAverage CI width by forecast year:")
for year in sorted(ci_analysis["Year"].unique()):
    avg_width = ci_analysis[ci_analysis["Year"] == year]["CI_Width"].mean()
    print(f"  {int(year)}: {avg_width:.1f}")

## 6. Key Takeaways

1. **ARIMA achieves a median MAPE of ~15.5%** on the 2020–2024 holdout — reasonable given this period includes the most volatile housing market in decades.

2. **Error is strongly correlated with pandemic-era housing boom intensity.** States like FL, NV, AZ, and ID that saw explosive price surges had the highest MAPE (>23%), while stable markets like DC, LA, and ND had MAPE under 8%.

3. **The model systematically underpredicts** in later holdout years (2022–2024), confirming that ARIMA — trained on pre-2020 trends — could not anticipate the accelerating post-COVID housing surge.

4. **Regional patterns are clear:** Sun Belt and Mountain West states are hardest to predict; Midwest and South Central states are the most accurate. This reflects fundamental differences in housing market dynamics: migration-driven booms vs. steady organic growth.

5. **Confidence intervals widen significantly** over the 10-year forecast horizon, reflecting growing uncertainty. By 2034, the average CI width is substantial — forecasts beyond 5 years should be treated as directional rather than precise.

6. **Limitations:** ARIMA is a univariate model — it cannot account for external factors like interest rates, migration patterns, housing supply, or policy changes. The 2025–2034 forecasts assume current trends continue, which is unlikely over a full decade.